# Semantic Similarity with `gemini-embedding-001`

## Setup

In [32]:
from google import genai
from google.genai import types
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [33]:
client = genai.Client()

## A Basic Test

More or less adapted from the example code on Google's documentation.

In [34]:
example_text1 = "I have a cute cat named Mouchan."
example_text2 = "My pet kitty Mouchan is so cute."
example_texts = [example_text1, example_text2]

In [35]:
def get_embeddings(texts: list[str]) -> types.EmbedContentResponse:
    return client.models.embed_content(
        model="gemini-embedding-001",
        contents=texts,
        config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY")
        )

In [36]:
result = get_embeddings(texts=example_texts)

In [49]:
df = pd.DataFrame(
    cosine_similarity([e.values for e in result.embeddings]),
    index=example_texts,
    columns=example_texts,
)

In [50]:
df

,I have a cute cat named Mouchan.,My pet kitty Mouchan is so cute.
I have a cute cat named Mouchan.,1.000000,0.958796
My pet kitty Mouchan is so cute.,0.958796,1.000000


## A More Sophsticated Test

Start with a series of texts, get their embeddings, and then check if a new text overlaps (has a high similarity score) to any of them.

In [51]:
example_texts2 = example_texts + [
    "I have a cute dog.",
    "I love my cat.",
    "You have to take your cat to the vet.",
    "Let's get tacos."
]

In [ ]:
def get_similarity(list_of_texts_to_check: types.EmbedContentResponse, new_text_to_check: types.EmbedContentResponse):
    # TODO: pass in original text (str) for clearer comparisons
    if list_of_texts_to_check.embeddings == None:
        raise ValueError("Results embeddings empty.")
    if new_text_to_check.embeddings == None:
        raise ValueError("New text embedding empty (None).")

    new_embedding_value = new_text_to_check.embeddings[0].values

    similarity_results = []

    for embedding in list_of_texts_to_check.embeddings:
        similarity = cosine_similarity([embedding.values, new_embedding_value])
        similarity_results.append(similarity)

    return similarity_results

In [55]:
more_results = get_embeddings(texts=example_texts2)
new_text_emb = get_embeddings(texts=["This is my cat."])
more_embeddings = get_similarity(list_of_texts_to_check=more_results, new_text_to_check=new_text_emb)

print(more_embeddings)

[array([[1.        , 0.90828491],
       [0.90828491, 1.        ]]), array([[1.        , 0.87630758],
       [0.87630758, 1.        ]]), array([[1.        , 0.84888982],
       [0.84888982, 1.        ]]), array([[1.       , 0.9225445],
       [0.9225445, 1.       ]]), array([[1.        , 0.84710723],
       [0.84710723, 1.        ]]), array([[1.        , 0.72686256],
       [0.72686256, 1.        ]])]
